# Grid Finding — Orchestrator (Houston / HCAD)

Runs the Grid Finding pipeline for Houston using Harris County Appraisal District (HCAD) building data. Notebooks 01–06 extract features, merge into a combined CSV, then run ML (07) and heatmap (08). Output folders: `csv/Houston/`, `outputs/Houston/`.

**Pipeline:**
```
01_grid_definition          → cell_id, zone_type (Y variable)
02_amenity_composition      → amenity_density, amenity_ratio_food_drink (OSM)
03_building_characteristics → avg_floors, avg_yearbuilt, building_count, total_bldg_area (HCAD)
04_land_use_mix             → landuse_entropy (HCAD)
05_tourism_intensity        → tourism_density (OSM)
06_commercial_density       → shop_density_km2, brand_ratio (OSM)
──────────────────────────────────────────────────────────────────────────────────────────────────────────
07_ml_classification        → train & evaluate models, export predictions
08_heatmap_visualization    → static + interactive heatmaps
```

**Usage:** Change parameters in the cell below → Restart kernel → Run All Cells

In [1]:
# ╔══════════════════════════════════════════════════════╗
# ║           PIPELINE PARAMETERS — CHANGE HERE         ║
# ╚══════════════════════════════════════════════════════╝

# ── Location ────────────────────────────────────────
# Houston, Texas using Harris County Appraisal District (HCAD) data.
# Single location analysis.

LOCATION = "Houston"

# ── Grid parameters ──────────────────────────────────
CELL_SIZE_M = 150            # grid cell size in meters
MIN_LOTS_PER_CELL = 3        # cells with fewer lots are dropped

# ── Classification mode ──────────────────────────────
INCLUDE_OTHER = False         # False = binary (Commercial vs Residential)
                              # True  = 3-class (+ Other)

# ── Combined sheets ──────────────────────────────────
COMBINED_SHEETS = False       # True  = generate combined ML sheets per plot
                              # False = skip (faster, comparison still runs)

# ── HCAD path ──────────────────────────────────────
# Path to Harris County Appraisal District (HCAD) building data CSV
HCAD_PATH = "hcad/harris_county_building_data.csv"  # Relative to ahmad/ folder

In [2]:
# ── Validate parameters + shared constants ────────────
import json
import os

if not os.path.exists(HCAD_PATH):
    raise FileNotFoundError(f"HCAD data not found at: {HCAD_PATH}\nUpdate HCAD_PATH in cell above.")

_ZONE_TYPE_RULES = {
    "Residential": {"landuse": "01", "threshold": 0.70},
    "Commercial": {"landuse": "02", "threshold": 0.50},
    "Industrial": {"landuse": "05", "threshold": 0.30},
    "Institutional": {"landuse": "08", "threshold": 0.30},
    "Open Space": {"landuse": ["09", "11"], "threshold": 0.30},
    "Mixed-Use": {"description": "No single category dominates"},
}

print(f"Location:        {LOCATION}")
print(f"HCAD data:       {HCAD_PATH}")
print(f"Cell size:       {CELL_SIZE_M}m")
print(f"Include Other:   {INCLUDE_OTHER}")

Location:        Houston
HCAD data:       hcad/harris_county_building_data.csv
Cell size:       150m
Include Other:   False


In [ ]:
import papermill as pm
import pandas as pd
import pathlib
import time
import os
import tempfile

PIPELINE = [
    ("01_grid_definition.ipynb",          "01 · Grid Definition"),
    ("02_amenity_composition.ipynb",      "02 · Amenity Composition"),
    ("03_building_characteristics.ipynb", "03 · Building Characteristics"),
    ("04_land_use_mix.ipynb",             "04 · Land Use Mix"),
    ("05_tourism_intensity.ipynb",        "05 · Tourism Intensity"),
    ("06_commercial_density.ipynb",       "06 · Commercial Density"),
]

KERNEL_NAME = "python3"

print(f"papermill {pm.__version__}")
print(f"Pipeline: {len(PIPELINE)} feature notebooks + ML + heatmap")

In [ ]:
# ── Helper: run a notebook via papermill ──────────────

def run_notebook(nb_path, description, params=None):
    """Run a notebook, return (status, elapsed)."""
    if not pathlib.Path(nb_path).exists():
        raise FileNotFoundError(f"Notebook not found: {nb_path}")
    tmp = pathlib.Path(tempfile.mktemp(suffix=".ipynb"))
    t0 = time.time()
    try:
        pm.execute_notebook(
            nb_path, str(tmp),
            kernel_name=KERNEL_NAME,
            parameters=params or {},
            progress_bar=False,
        )
        elapsed = time.time() - t0
        print(f"  OK  ({elapsed:.1f} s)")
        return "ok", elapsed
    except pm.exceptions.PapermillExecutionError as e:
        elapsed = time.time() - t0
        print(f"  FAILED  ({elapsed:.1f} s)")
        print(f"  Error: {e}")
        return "failed", elapsed
    finally:
        if tmp.exists():
            tmp.unlink()

In [ ]:
# ══════════════════════════════════════════════════════
#  MAIN LOOP — run full pipeline for Houston
# ══════════════════════════════════════════════════════

all_results = {}

csv_dir = f"csv/{LOCATION}"
plots_dir = f"outputs/{LOCATION}"

# ── Clean only ML output files (keep feature CSVs) ────
ml_output_files = [
    f"{csv_dir}/07_predictions.csv",
]

for fpath in ml_output_files:
    p = pathlib.Path(fpath)
    if p.exists():
        p.unlink()
        print(f"  Deleted: {fpath}")

# Clean plots directory only
plots_p = pathlib.Path(plots_dir)
if plots_p.exists():
    for f in plots_p.iterdir():
        if f.is_file():
            f.unlink()
else:
    plots_p.mkdir(parents=True, exist_ok=True)

# Ensure csv_dir exists
pathlib.Path(csv_dir).mkdir(parents=True, exist_ok=True)

print(f"\n{'#'*60}")
print(f"  LOCATION: {LOCATION}")
print(f"  CSV:      {csv_dir}/")
print(f"  Plots:    {plots_dir}/")
print(f"{'#'*60}")

# ── Write grid.json for Houston ────────────────────
config = {
    "location": LOCATION,
    "grid_cell_size_m": CELL_SIZE_M,
    "min_lots_per_cell": MIN_LOTS_PER_CELL,
    "hcad_path": HCAD_PATH,
    "include_other": INCLUDE_OTHER,
    "csv_dir": csv_dir,
    "feature_flags": {"needs_hcad": True},
    "zone_type_rules": _ZONE_TYPE_RULES,
}
with open("grid.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=4)

# ── Run feature extraction notebooks (01–06) ─────
results = []
for nb_path, description in PIPELINE:
    print(f"\n  {description}")
    status, elapsed = run_notebook(
        nb_path, description,
        params={},  # Notebooks read grid.json directly, no parameters needed
    )
    results.append((nb_path, status, elapsed))

failed = [nb for nb, s, _ in results if s == "failed"]
if failed:
    print(f"\n  WARNING: {len(failed)} notebook(s) failed: {failed}")

# ── Merge CSVs ───────────────────────────────────
csv_files = [
    f"{csv_dir}/01_grid_definition.csv",
    f"{csv_dir}/02_amenity_composition.csv",
    f"{csv_dir}/03_building_characteristics.csv",
    f"{csv_dir}/04_land_use_mix.csv",
    f"{csv_dir}/05_tourism_intensity.csv",
    f"{csv_dir}/06_commercial_density.csv",
]

df_combined = pd.read_csv(csv_files[0], dtype={"cell_id": str})
for csv_path in csv_files[1:]:
    if pathlib.Path(csv_path).exists():
        df_other = pd.read_csv(csv_path, dtype={"cell_id": str})
        merge_cols = [c for c in df_other.columns if c != "cell_id"]
        df_combined = df_combined.merge(
            df_other[["cell_id"] + merge_cols],
            on="cell_id", how="left",
        )

combined_path = f"{csv_dir}/combined_grid.csv"
df_combined.to_csv(combined_path, index=False, encoding="utf-8")
print(f"\n  Combined: {df_combined.shape[0]} cells x {df_combined.shape[1]} cols")

# ── Run ML (07) ──────────────────────────────────
print(f"\n  07 · ML Classification")
run_notebook(
    "07_ml_classification.ipynb", "ML",
    params={"CSV_PATH": combined_path, "PLOTS_DIR": plots_dir},
)

# ── Run Heatmap (08) ─────────────────────────────
print(f"\n  08 · Heatmap Visualization")
run_notebook(
    "08_heatmap_visualization.ipynb", "Heatmap",
    params={"PLOTS_DIR": plots_dir},
)

all_results[LOCATION] = {
    "cells": len(df_combined),
    "csv_dir": csv_dir,
    "plots_dir": plots_dir,
    "results": results,
}

print(f"\n  {LOCATION} DONE — {len(df_combined)} cells")

NameError: name 'pathlib' is not defined

In [ ]:
# ── Summary ──────────────────────────────────────────
print(f"\n{'#'*60}")
print(f"  HOUSTON GRID FINDING PIPELINE COMPLETE")
print(f"{'#'*60}")
for name, info in all_results.items():
    print(f"  {name}: {info['plots_dir']}/")
print()